# Minimal placeholder pipeline for bond valuation with embedded options

This notebook reads a CSV placeholder from the `data/` folder, constructs a tiny bond specification, and values the embedded option with the repository's Hull-White multi-curve tree.


In [ ]:
from pathlib import Path

import pandas as pd

from src.curve import YieldCurve
from src.multi_hw_tree import BondSpec, CouponDef, ExerciseSpec, MultiCurveHWTree

root = Path.cwd().resolve().parent
placeholder = pd.read_csv(root / 'data' / 'bonds_placeholder.csv')
display(placeholder)

curve = YieldCurve.from_zero_rates(
    maturities=[1.0, 2.0, 3.0, 4.0, 5.0],
    zero_rates=[0.031, 0.032, 0.033, 0.034, 0.035],
)

tree = MultiCurveHWTree(
    a_r=0.03,
    sigma_r=0.015,
    a_L=0.02,
    sigma_L=0.018,
    rho=0.20,
    disc_curve=curve,
    ref_curve=curve,
    dt=1.0,
    n_steps=4,
)

coupon_map = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

bond_european = BondSpec(face=100.0, maturity_step=4, coupons=coupon_map)
exercise_european = ExerciseSpec(call={4: 101.0})

bond_bermudan = BondSpec(face=100.0, maturity_step=4, coupons=coupon_map)
exercise_bermudan = ExerciseSpec(put={2: 99.0, 4: 99.0})

results = {
    'bond_type': ['European call', 'Bermudan put'],
    'full_price': [
        tree.price(bond_european, exercise_european),
        tree.price(bond_bermudan, exercise_bermudan),
    ],
    'straight_price': [
        tree.price(bond_european),
        tree.price(bond_bermudan),
    ],
}

pd.DataFrame(results)
